In [1]:
pip install numpy pandas scikit-learn keras


Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# Function to load and preprocess


In [8]:
#Random Forest
import os
import pandas as pd
import optuna
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Function to load and preprocess data
def load_and_preprocess_data(folder_path):
    columns = [
        "Time",
        "L1", "L2", "L3", "L4", "L5", "L6", "L7", "L8",
        "R1", "R2", "R3", "R4", "R5", "R6", "R7", "R8",
        "TotalForceLeft", "TotalForceRight",
        "Label"
    ]

    all_data_combined = pd.DataFrame()

    for file_name in os.listdir(folder_path):
        if file_name.endswith(".txt"):
            file_path = os.path.join(folder_path, file_name)

            df = pd.read_csv(file_path, header=None, delimiter='\s+', names=columns)
            df = df.drop(columns=["Time"])

            df["Label"] = 1 if "Pt" in file_name else 0

            all_data_combined = pd.concat([all_data_combined, df], ignore_index=True)

    all_data_combined = all_data_combined.apply(pd.to_numeric, errors='coerce')

    return all_data_combined

# Load and preprocess data
folder_path =  r"D:\Newfolder\s7\project\gait dataset\gait-in-parkinsons-disease-1.0.0\data" # Replace with the actual path
data = load_and_preprocess_data(folder_path)

# Split data into features and labels
X = data.drop(columns=["Label"])
y = data["Label"]

# Split the data into training and testing sets
X_train, X_test, y_train,y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Impute missing values
imputer = SimpleImputer(strategy='mean')
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

# Standardize features
scaler = StandardScaler()
X_train_imputed_scaled = scaler.fit_transform(X_train_imputed)
X_test_imputed_scaled = scaler.transform(X_test_imputed)

# Define the objective function for optimization
def objective(trial):
    # Define hyperparameters to be optimized
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 4)

    # Create a RandomForestClassifier with hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        random_state=42
    )

    # Train the model
    model.fit(X_train_imputed_scaled, y_train)

    # Make predictions on the test set
    y_pred = model.predict(X_test_imputed_scaled)

    # Evaluate the model using accuracy
    accuracy = accuracy_score(y_test, y_pred)
    return accuracy

# Create an Optuna study and run the optimization process
study = optuna.create_study(direction='maximize')  # maximize accuracy
study.optimize(objective, n_trials=5)

# Get the best hyperparameters
best_params = study.best_params
print("Best Hyperparameters:", best_params)

# Train the final model with the best hyperparameters
best_model = RandomForestClassifier(
    n_estimators=best_params['n_estimators'],
    max_depth=best_params['max_depth'],
    min_samples_split=best_params['min_samples_split'],
    min_samples_leaf=best_params['min_samples_leaf'],
    random_state=42
)

best_model.fit(X_train_imputed_scaled, y_train)

# Make predictions on the test set using the best model
y_pred = best_model.predict(X_test_imputed_scaled)

# Evaluate the final model
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1 Score: {f1:.2f}")
print("Confusion Matrix:")
print(conf_matrix)



[I 2023-12-13 13:57:12,313] A new study created in memory with name: no-name-3c64ced1-f286-4ce8-861e-1092cc706b11
[I 2023-12-13 14:16:11,309] Trial 0 finished with value: 0.80088701619867 and parameters: {'n_estimators': 115, 'max_depth': 10, 'min_samples_split': 4, 'min_samples_leaf': 4}. Best is trial 0 with value: 0.80088701619867.
[I 2023-12-13 14:27:58,621] Trial 1 finished with value: 0.7653280046053712 and parameters: {'n_estimators': 144, 'max_depth': 8, 'min_samples_split': 8, 'min_samples_leaf': 1}. Best is trial 0 with value: 0.80088701619867.
[I 2023-12-13 14:45:11,375] Trial 2 finished with value: 0.8551749393058488 and parameters: {'n_estimators': 165, 'max_depth': 13, 'min_samples_split': 6, 'min_samples_leaf': 1}. Best is trial 2 with value: 0.8551749393058488.
[I 2023-12-13 14:48:13,022] Trial 3 finished with value: 0.7218844045794509 and parameters: {'n_estimators': 73, 'max_depth': 5, 'min_samples_split': 2, 'min_samples_leaf': 4}. Best is trial 2 with value: 0.85517

Best Hyperparameters: {'n_estimators': 165, 'max_depth': 13, 'min_samples_split': 6, 'min_samples_leaf': 1}
Accuracy: 0.86
Precision: 0.83
Recall: 0.99
F1 Score: 0.90
Confusion Matrix:
[[112846  90074]
 [  6028 454625]]


In [1]:
#KNN
import os
import pandas as pd
import optuna
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Function to load and preprocess data
def load_and_preprocess_data(folder_path):
    columns = [
        "Time",
        "L1", "L2", "L3", "L4", "L5", "L6", "L7", "L8",
        "R1", "R2", "R3", "R4", "R5", "R6", "R7", "R8",
        "TotalForceLeft", "TotalForceRight",
        "Label"
    ]

    all_data_combined = pd.DataFrame()

    for file_name in os.listdir(folder_path):
        if file_name.endswith(".txt"):
            file_path = os.path.join(folder_path, file_name)

            df = pd.read_csv(file_path, header=None, delimiter='\s+', names=columns)
            df = df.drop(columns=["Time"])

            df["Label"] = 1 if "Pt" in file_name else 0

            all_data_combined = pd.concat([all_data_combined, df], ignore_index=True)

    all_data_combined = all_data_combined.apply(pd.to_numeric, errors='coerce')

    return all_data_combined

# Load and preprocess data
folder_path =  r"D:\Newfolder\s7\project\gait dataset\gait-in-parkinsons-disease-1.0.0\data" # Replace with the actual path
data = load_and_preprocess_data(folder_path)

# Split data into features and labels
X = data.drop(columns=["Label"])
y = data["Label"]

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Impute missing values
imputer = SimpleImputer(strategy='mean')
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

# Standardize features
scaler = StandardScaler()
X_train_imputed_scaled = scaler.fit_transform(X_train_imputed)
X_test_imputed_scaled = scaler.transform(X_test_imputed)

# Define the objective function for optimization
def objective(trial):
    # Define hyperparameters to be optimized
    n_neighbors = trial.suggest_int('n_neighbors', 1, 20)
    # You can include other hyperparameters specific to KNN

    # Create a KNeighborsClassifier with hyperparameters
    model = KNeighborsClassifier(
        n_neighbors=n_neighbors,
        # Include other hyperparameters here
    )

    # Train the model
    model.fit(X_train_imputed_scaled, y_train)

    # Make predictions on the test set
    y_pred = model.predict(X_test_imputed_scaled)

    # Evaluate the model using accuracy
    accuracy = accuracy_score(y_test, y_pred)
    return accuracy

# Create an Optuna study and run the optimization process
study = optuna.create_study(direction='maximize')  # maximize accuracy
study.optimize(objective, n_trials=5)

# Get the best hyperparameters
best_params = study.best_params
print("Best Hyperparameters:", best_params)

# Train the final model with the best hyperparameters
best_model = KNeighborsClassifier(
    n_neighbors=best_params['n_neighbors'],
    # Include other best hyperparameters here
)

best_model.fit(X_train_imputed_scaled, y_train)

# Make predictions on the test set using the best model
y_pred = best_model.predict(X_test_imputed_scaled)

# Evaluate the final model
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1 Score: {f1:.2f}")
print("Confusion Matrix:")
print(conf_matrix)


[I 2023-12-14 12:22:18,839] A new study created in memory with name: no-name-6bcc22a9-c22f-46fe-a4f1-76348b85d323
[I 2023-12-14 13:34:30,001] Trial 0 finished with value: 0.9838390049022488 and parameters: {'n_neighbors': 5}. Best is trial 0 with value: 0.9838390049022488.
[I 2023-12-14 14:43:39,028] Trial 1 finished with value: 0.9818136060388232 and parameters: {'n_neighbors': 7}. Best is trial 0 with value: 0.9838390049022488.
[I 2023-12-14 15:59:01,998] Trial 2 finished with value: 0.9838390049022488 and parameters: {'n_neighbors': 5}. Best is trial 0 with value: 0.9838390049022488.
[I 2023-12-14 17:33:19,274] Trial 3 finished with value: 0.9784078014024079 and parameters: {'n_neighbors': 11}. Best is trial 0 with value: 0.9838390049022488.
[I 2023-12-14 18:40:00,341] Trial 4 finished with value: 0.9799931582508631 and parameters: {'n_neighbors': 9}. Best is trial 0 with value: 0.9838390049022488.


Best Hyperparameters: {'n_neighbors': 5}
Accuracy: 0.98
Precision: 0.99
Recall: 0.99
F1 Score: 0.99
Confusion Matrix:
[[196297   6623]
 [  4101 456552]]


In [6]:
#Decision Tree 
import os
import pandas as pd
import optuna
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Function to load and preprocess data
def load_and_preprocess_data(folder_path):
    columns = [
        "Time",
        "L1", "L2", "L3", "L4", "L5", "L6", "L7", "L8",
        "R1", "R2", "R3", "R4", "R5", "R6", "R7", "R8",
        "TotalForceLeft", "TotalForceRight",
        "Label"
    ]

    all_data_combined = pd.DataFrame()

    for file_name in os.listdir(folder_path):
        if file_name.endswith(".txt"):
            file_path = os.path.join(folder_path, file_name)

            df = pd.read_csv(file_path, header=None, delimiter='\s+', names=columns)
            df = df.drop(columns=["Time"])

            df["Label"] = 1 if "Pt" in file_name else 0

            all_data_combined = pd.concat([all_data_combined, df], ignore_index=True)

    all_data_combined = all_data_combined.apply(pd.to_numeric, errors='coerce')

    return all_data_combined

# Load and preprocess data
folder_path = r"D:\Newfolder\s7\project\gait dataset\gait-in-parkinsons-disease-1.0.0\data"  # Replace with the actual path
data = load_and_preprocess_data(folder_path)

# Split data into features and labels
X = data.drop(columns=["Label"])
y = data["Label"]

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Impute missing values
imputer = SimpleImputer(strategy='mean')
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

# Standardize features
scaler = StandardScaler()
X_train_imputed_scaled = scaler.fit_transform(X_train_imputed)
X_test_imputed_scaled = scaler.transform(X_test_imputed)

# Define the objective function for optimization
def objective(trial):
    # Define hyperparameters to be optimized
    criterion = trial.suggest_categorical('criterion', ['gini', 'entropy'])
    splitter = trial.suggest_categorical('splitter', ['best', 'random'])
    max_depth = trial.suggest_int('max_depth', 3, 20)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 4)
    max_features = trial.suggest_categorical('max_features', ['auto', 'sqrt', 'log2', None])
    max_leaf_nodes = trial.suggest_int('max_leaf_nodes', 10, 100)
    min_impurity_decrease = trial.suggest_float('min_impurity_decrease', 0.0, 0.5)
    min_weight_fraction_leaf = trial.suggest_float('min_weight_fraction_leaf', 0.0, 0.5)

    # Create a DecisionTreeClassifier with hyperparameters
    model = DecisionTreeClassifier(
        criterion=criterion,
        splitter=splitter,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        max_leaf_nodes=max_leaf_nodes,
        min_impurity_decrease=min_impurity_decrease,
        min_weight_fraction_leaf=min_weight_fraction_leaf,
        random_state=42
    )

    # Train the model
    model.fit(X_train_imputed_scaled, y_train)

    # Make predictions on the test set
    y_pred = model.predict(X_test_imputed_scaled)

    # Evaluate the model using accuracy
    accuracy = accuracy_score(y_test, y_pred)
    return accuracy

# Create an Optuna study and run the optimization process
study = optuna.create_study(direction='maximize')  # maximize accuracy
study.optimize(objective, n_trials=5)

# Get the best hyperparameters
best_params = study.best_params
print("Best Hyperparameters:", best_params)

# Train the final model with the best hyperparameters
best_model = DecisionTreeClassifier(
    criterion=best_params['criterion'],
    splitter=best_params['splitter'],
    max_depth=best_params['max_depth'],
    min_samples_split=best_params['min_samples_split'],
    min_samples_leaf=best_params['min_samples_leaf'],
    max_features=best_params['max_features'],
    max_leaf_nodes=best_params['max_leaf_nodes'],
    min_impurity_decrease=best_params['min_impurity_decrease'],
    min_weight_fraction_leaf=best_params['min_weight_fraction_leaf'],
    random_state=42
)

best_model.fit(X_train_imputed_scaled, y_train)

# Make predictions on the test set using the best model
y_pred = best_model.predict(X_test_imputed_scaled)

# Evaluate the final model
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1 Score: {f1:.2f}")
print("Confusion Matrix:")
print(conf_matrix)


[I 2023-12-13 19:42:36,587] A new study created in memory with name: no-name-bf1a9118-f4d2-48a4-b99b-c084bd18f777
[I 2023-12-13 19:42:38,232] Trial 0 finished with value: 0.6942009394595621 and parameters: {'criterion': 'entropy', 'splitter': 'best', 'max_depth': 13, 'min_samples_split': 3, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_leaf_nodes': 90, 'min_impurity_decrease': 0.3626568127050697, 'min_weight_fraction_leaf': 0.332969162050447}. Best is trial 0 with value: 0.6942009394595621.
[I 2023-12-13 19:42:39,491] Trial 1 finished with value: 0.6942009394595621 and parameters: {'criterion': 'entropy', 'splitter': 'best', 'max_depth': 15, 'min_samples_split': 2, 'min_samples_leaf': 3, 'max_features': 'log2', 'max_leaf_nodes': 53, 'min_impurity_decrease': 0.392369643883813, 'min_weight_fraction_leaf': 0.03277951013394903}. Best is trial 0 with value: 0.6942009394595621.
[I 2023-12-13 19:42:40,715] Trial 2 finished with value: 0.6942009394595621 and parameters: {'criterion': 'gi

Best Hyperparameters: {'criterion': 'entropy', 'splitter': 'best', 'max_depth': 13, 'min_samples_split': 3, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_leaf_nodes': 90, 'min_impurity_decrease': 0.3626568127050697, 'min_weight_fraction_leaf': 0.332969162050447}
Accuracy: 0.69
Precision: 0.69
Recall: 1.00
F1 Score: 0.82
Confusion Matrix:
[[     0 202920]
 [     0 460653]]


In [7]:
#Naive Bayes
import os
import pandas as pd
import optuna
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Function to load and preprocess data
def load_and_preprocess_data(folder_path):
    columns = [
        "Time",
        "L1", "L2", "L3", "L4", "L5", "L6", "L7", "L8",
        "R1", "R2", "R3", "R4", "R5", "R6", "R7", "R8",
        "TotalForceLeft", "TotalForceRight",
        "Label"
    ]

    all_data_combined = pd.DataFrame()

    for file_name in os.listdir(folder_path):
        if file_name.endswith(".txt"):
            file_path = os.path.join(folder_path, file_name)

            df = pd.read_csv(file_path, header=None, delimiter='\s+', names=columns)
            df = df.drop(columns=["Time"])

            df["Label"] = 1 if "Pt" in file_name else 0

            all_data_combined = pd.concat([all_data_combined, df], ignore_index=True)

    all_data_combined = all_data_combined.apply(pd.to_numeric, errors='coerce')

    return all_data_combined

# Load and preprocess data
folder_path = r"D:\Newfolder\s7\project\gait dataset\gait-in-parkinsons-disease-1.0.0\data"  # Replace with the actual path
data = load_and_preprocess_data(folder_path)

# Split data into features and labels
X = data.drop(columns=["Label"])
y = data["Label"]

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Impute missing values
imputer = SimpleImputer(strategy='mean')
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

# Standardize features
scaler = StandardScaler()
X_train_imputed_scaled = scaler.fit_transform(X_train_imputed)
X_test_imputed_scaled = scaler.transform(X_test_imputed)

# Define the objective function for optimization
def objective(trial):
    # No specific hyperparameters to tune for Gaussian Naive Bayes
    model = GaussianNB()
    
    # Train the model
    model.fit(X_train_imputed_scaled, y_train)

    # Make predictions on the test set
    y_pred = model.predict(X_test_imputed_scaled)

    # Evaluate the model using accuracy
    accuracy = accuracy_score(y_test, y_pred)
    return accuracy

# Create an Optuna study and run the optimization process
study = optuna.create_study(direction='maximize')  # maximize accuracy
study.optimize(objective, n_trials=5)

# Get the best hyperparameters (there are none for Gaussian Naive Bayes)
best_params = study.best_params
print("Best Hyperparameters:", best_params)

# Train the final model (no hyperparameters to set for Gaussian Naive Bayes)
best_model = GaussianNB()
best_model.fit(X_train_imputed_scaled, y_train)

# Make predictions on the test set using the best model
y_pred = best_model.predict(X_test_imputed_scaled)

# Evaluate the final model
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1 Score: {f1:.2f}")
print("Confusion Matrix:")
print(conf_matrix)


[I 2023-12-13 19:50:21,718] A new study created in memory with name: no-name-1f427ef2-f1f5-48e7-994a-bfd3ec81ccfb
[I 2023-12-13 19:50:22,963] Trial 0 finished with value: 0.6769413463175867 and parameters: {}. Best is trial 0 with value: 0.6769413463175867.
[I 2023-12-13 19:50:24,095] Trial 1 finished with value: 0.6769413463175867 and parameters: {}. Best is trial 0 with value: 0.6769413463175867.
[I 2023-12-13 19:50:25,172] Trial 2 finished with value: 0.6769413463175867 and parameters: {}. Best is trial 0 with value: 0.6769413463175867.
[I 2023-12-13 19:50:26,266] Trial 3 finished with value: 0.6769413463175867 and parameters: {}. Best is trial 0 with value: 0.6769413463175867.
[I 2023-12-13 19:50:27,334] Trial 4 finished with value: 0.6769413463175867 and parameters: {}. Best is trial 0 with value: 0.6769413463175867.


Best Hyperparameters: {}
Accuracy: 0.68
Precision: 0.72
Recall: 0.87
F1 Score: 0.79
Confusion Matrix:
[[ 46633 156287]
 [ 58086 402567]]


In [2]:
#Multi Layer Perceptron
import os
import pandas as pd
import optuna
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Function to load and preprocess data
def load_and_preprocess_data(folder_path):
    columns = [
        "Time",
        "L1", "L2", "L3", "L4", "L5", "L6", "L7", "L8",
        "R1", "R2", "R3", "R4", "R5", "R6", "R7", "R8",
        "TotalForceLeft", "TotalForceRight",
        "Label"
    ]

    all_data_combined = pd.DataFrame()

    for file_name in os.listdir(folder_path):
        if file_name.endswith(".txt"):
            file_path = os.path.join(folder_path, file_name)

            df = pd.read_csv(file_path, header=None, delimiter='\s+', names=columns)
            df = df.drop(columns=["Time"])

            df["Label"] = 1 if "Pt" in file_name else 0

            all_data_combined = pd.concat([all_data_combined, df], ignore_index=True)

    all_data_combined = all_data_combined.apply(pd.to_numeric, errors='coerce')

    return all_data_combined

# Load and preprocess data
folder_path = r"D:\Newfolder\s7\project\gait dataset\gait-in-parkinsons-disease-1.0.0\data"  # Replace with the actual path
data = load_and_preprocess_data(folder_path)

# Split data into features and labels
X = data.drop(columns=["Label"])
y = data["Label"]

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Impute missing values
imputer = SimpleImputer(strategy='mean')
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

# Standardize features
scaler = StandardScaler()
X_train_imputed_scaled = scaler.fit_transform(X_train_imputed)
X_test_imputed_scaled = scaler.transform(X_test_imputed)

# Define the objective function for optimization
def objective(trial):
    # Define hyperparameters to be optimized
    layer1_size = trial.suggest_int('layer1_size', 50, 100)
    layer2_size = trial.suggest_int('layer2_size', 50, 100)
    layer3_size = trial.suggest_int('layer3_size', 50, 100)
    hidden_layer_sizes = (layer1_size, layer2_size, layer3_size)

    activation = trial.suggest_categorical('activation', ['relu', 'tanh', 'logistic'])
    solver = trial.suggest_categorical('solver', ['adam', 'sgd'])
    learning_rate = trial.suggest_categorical('learning_rate', ['constant', 'invscaling', 'adaptive'])
    max_iter = trial.suggest_int('max_iter', 50, 200)
    early_stopping = trial.suggest_categorical('early_stopping', [True, False])
    validation_fraction = trial.suggest_float('validation_fraction', 0.1, 0.3)
    patience = trial.suggest_int('patience', 5, 20)

    # Create an MLPClassifier with hyperparameters
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layer_sizes,
        activation=activation,
        solver=solver,
        learning_rate=learning_rate,
        max_iter=max_iter,
        early_stopping=early_stopping,
        validation_fraction=validation_fraction,
        n_iter_no_change=patience,
        random_state=42
    )

    # Train the model
    model.fit(X_train_imputed_scaled, y_train)

    # Make predictions on the test set
    y_pred = model.predict(X_test_imputed_scaled)

    # Evaluate the model using accuracy
    accuracy = accuracy_score(y_test, y_pred)
    return accuracy

# Create an Optuna study and run the optimization process
study = optuna.create_study(direction='maximize')  # maximize accuracy
study.optimize(objective, n_trials=3)

# Get the best hyperparameters
best_params = study.best_params
print("Best Hyperparameters:", best_params)

# Train the final model with the best hyperparameters
best_model = MLPClassifier(
    hidden_layer_sizes=(
    best_params['layer1_size'],
    best_params['layer2_size'],
    best_params['layer3_size'],
),
activation=best_params['activation'],

    solver=best_params['solver'],
    learning_rate=best_params['learning_rate'],
    max_iter=best_params['max_iter'],
    early_stopping=best_params['early_stopping'],
    validation_fraction=best_params['validation_fraction'],
    n_iter_no_change=best_params['patience'],
    random_state=42
)

best_model.fit(X_train_imputed_scaled, y_train)

# Make predictions on the test set using the best model
y_pred = best_model.predict(X_test_imputed_scaled)

# Evaluate the final model
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)




print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1 Score: {f1:.2f}")
print("Confusion Matrix:")
print(conf_matrix)



[I 2023-12-14 21:05:06,518] A new study created in memory with name: no-name-454a9ec0-0338-4249-af88-a593f9af869b
C:\Users\Asus\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (112) reached and the optimization hasn't converged yet.
  warnings.warn(
[I 2023-12-14 22:12:55,574] Trial 0 finished with value: 0.9556537110461095 and parameters: {'layer1_size': 58, 'layer2_size': 50, 'layer3_size': 94, 'activation': 'relu', 'solver': 'sgd', 'learning_rate': 'adaptive', 'max_iter': 112, 'early_stopping': True, 'validation_fraction': 0.18041930956080482, 'patience': 19}. Best is trial 0 with value: 0.9556537110461095.
[I 2023-12-14 23:18:27,527] Trial 1 finished with value: 0.9773574271406462 and parameters: {'layer1_size': 74, 'layer2_size': 96, 'layer3_size': 99, 'activation': 'logistic', 'solver': 'adam', 'learning_rate': 'constant', 'max_iter': 171, 'early_stopping': True, 'validation_fraction': 

Best Hyperparameters: {'layer1_size': 74, 'layer2_size': 96, 'layer3_size': 99, 'activation': 'logistic', 'solver': 'adam', 'learning_rate': 'constant', 'max_iter': 171, 'early_stopping': True, 'validation_fraction': 0.19737548553228032, 'patience': 9}
Accuracy: 0.98
Precision: 0.98
Recall: 0.99
F1 Score: 0.98
Confusion Matrix:
[[193977   8943]
 [  6082 454571]]


In [15]:
#Logistic Regression
import os
import pandas as pd
import optuna
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Function to load and preprocess data
def load_and_preprocess_data(folder_path):
    columns = [
        "Time",
        "L1", "L2", "L3", "L4", "L5", "L6", "L7", "L8",
        "R1", "R2", "R3", "R4", "R5", "R6", "R7", "R8",
        "TotalForceLeft", "TotalForceRight",
        "Label"
    ]

    all_data_combined = pd.DataFrame()

    for file_name in os.listdir(folder_path):
        if file_name.endswith(".txt"):
            file_path = os.path.join(folder_path, file_name)

            df = pd.read_csv(file_path, header=None, delimiter='\s+', names=columns)
            df = df.drop(columns=["Time"])

            df["Label"] = 1 if "Pt" in file_name else 0

            all_data_combined = pd.concat([all_data_combined, df], ignore_index=True)

    all_data_combined = all_data_combined.apply(pd.to_numeric, errors='coerce')

    return all_data_combined

# Load and preprocess data
folder_path = r"D:\Newfolder\s7\project\gait dataset\gait-in-parkinsons-disease-1.0.0\data"  # Replace with the actual path
data = load_and_preprocess_data(folder_path)

# Split data into features and labels
X = data.drop(columns=["Label"])
y = data["Label"]

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Impute missing values
imputer = SimpleImputer(strategy='mean')
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

# Standardize features
scaler = StandardScaler()
X_train_imputed_scaled = scaler.fit_transform(X_train_imputed)
X_test_imputed_scaled = scaler.transform(X_test_imputed)

# Define the objective function for optimization
def objective(trial):
    # Define hyperparameters to be optimized
    penalty = trial.suggest_categorical('penalty', ['l2'])  # Change to 'l2'
    C = trial.suggest_loguniform('C', 1e-4, 1e4)
    solver = trial.suggest_categorical('solver', ['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga'])
    max_iter = trial.suggest_int('max_iter', 100, 1000)
    multi_class = 'ovr'
    class_weight = trial.suggest_categorical('class_weight', [None, 'balanced'])

    # Create a LogisticRegression model with hyperparameters
    model = LogisticRegression(
        penalty=penalty,
        C=C,
        solver=solver,
        max_iter=max_iter,
        multi_class=multi_class,
        class_weight=class_weight,
        random_state=42
    )

    # Train the model
    model.fit(X_train_imputed_scaled, y_train)

    # Make predictions on the test set
    y_pred = model.predict(X_test_imputed_scaled)

    # Evaluate the model using accuracy
    accuracy = accuracy_score(y_test, y_pred)
    return accuracy


# Create an Optuna study and run the optimization process
study = optuna.create_study(direction='maximize')  # maximize accuracy
study.optimize(objective, n_trials=5)

# Get the best hyperparameters
best_params = study.best_params
print("Best Hyperparameters:", best_params)

 # Train the final model with the best hyperparameters
best_model = LogisticRegression(
    penalty=best_params['penalty'],
    C=best_params['C'],
    solver=best_params['solver'],
    max_iter=best_params['max_iter'],
    class_weight=best_params['class_weight'],
    random_state=42
)

best_model.fit(X_train_imputed_scaled, y_train)
# Make predictions on the test set using the best model
y_pred = best_model.predict(X_test_imputed_scaled)


# Evaluate the final model
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1 Score: {f1:.2f}")
print("Confusion Matrix:")
print(conf_matrix)


[I 2023-12-13 23:45:21,934] A new study created in memory with name: no-name-75a98ff7-feb9-4f52-9a65-9d99b7909602
C:\Users\Asus\AppData\Local\Temp\ipykernel_12884\988585229.py:62: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C = trial.suggest_loguniform('C', 1e-4, 1e4)
[I 2023-12-13 23:46:12,448] Trial 0 finished with value: 0.5590146072851065 and parameters: {'penalty': 'l2', 'C': 0.00042854122455553644, 'solver': 'sag', 'max_iter': 538, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.5590146072851065.
C:\Users\Asus\AppData\Local\Temp\ipykernel_12884\988585229.py:62: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C = trial.suggest_loguniform('C', 1e-4, 1e4)


Best Hyperparameters: {'penalty': 'l2', 'C': 517.476632279489, 'solver': 'liblinear', 'max_iter': 456, 'class_weight': None}
Accuracy: 0.70
Precision: 0.70
Recall: 1.00
F1 Score: 0.82
Confusion Matrix:
[[  2730 200190]
 [   645 460008]]


In [ ]:
#svm
import os
import pandas as pd
import optuna
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Function to load and preprocess data
def load_and_preprocess_data(folder_path):
    columns = [
        "Time",
        "L1", "L2", "L3", "L4", "L5", "L6", "L7", "L8",
        "R1", "R2", "R3", "R4", "R5", "R6", "R7", "R8",
        "TotalForceLeft", "TotalForceRight",
        "Label"
    ]

    all_data_combined = pd.DataFrame()

    for file_name in os.listdir(folder_path):
        if file_name.endswith(".txt"):
            file_path = os.path.join(folder_path, file_name)

            df = pd.read_csv(file_path, header=None, delimiter='\s+', names=columns)
            df = df.drop(columns=["Time"])

            df["Label"] = 1 if "Pt" in file_name else 0

            all_data_combined = pd.concat([all_data_combined, df], ignore_index=True)

    all_data_combined = all_data_combined.apply(pd.to_numeric, errors='coerce')

    return all_data_combined
    # ... (unchanged)

# Load and preprocess data
folder_path =  r"D:\Newfolder\s7\project\gait dataset\gait-in-parkinsons-disease-1.0.0\data"
data = load_and_preprocess_data(folder_path)

# Split data into features and labels
X = data.drop(columns=["Label"])
y = data["Label"]

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Impute missing values
imputer = SimpleImputer(strategy='mean')
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

# Standardize features
scaler = StandardScaler()
X_train_imputed_scaled = scaler.fit_transform(X_train_imputed)
X_test_imputed_scaled = scaler.transform(X_test_imputed)

# Define the objective function for optimization
def objective(trial):
    # Define hyperparameters to be optimized
    C = trial.suggest_float('C', 1e-3, 1e3, log=True)  # Regularization parameter with log scale
    kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly'])  # Kernel type
    gamma = trial.suggest_float('gamma', 1e-4, 1e3, log=True) if kernel in ['rbf', 'poly'] else 'scale'
    degree = trial.suggest_int('degree', 2, 5) if kernel == 'poly' else 3  # Degree of the polynomial kernel
    class_weight = trial.suggest_categorical('class_weight', [None, 'balanced'])  # Class weights

    # Create an SVM model with hyperparameters
    model = SVC(
        C=C,
        kernel=kernel,
        gamma=gamma,
        degree=degree,
        class_weight=class_weight,
        random_state=42
    )

    # Train the model
    model.fit(X_train_imputed_scaled, y_train)

    # Make predictions on the test set
    y_pred = model.predict(X_test_imputed_scaled)

    # Evaluate the model using accuracy
    accuracy = accuracy_score(y_test, y_pred)
    return accuracy

# Create an Optuna study and run the optimization process
study = optuna.create_study(direction='maximize')  # maximize accuracy
study.optimize(objective, n_trials=5)

# Get the best hyperparameters
best_params = study.best_params
print("Best Hyperparameters:", best_params)

# Train the final SVM model with the best hyperparameters
best_svm_model = SVC(
    C=best_params['C'],
    kernel=best_params['kernel'],
    gamma=best_params['gamma'],
    degree=best_params['degree'],
    class_weight=best_params['class_weight'],
    random_state=42
)

best_svm_model.fit(X_train_imputed_scaled, y_train)

# Make predictions on the test set using the best model
y_pred_svm = best_svm_model.predict(X_test_imputed_scaled)

# Evaluate the final SVM model
accuracy_svm = accuracy_score(y_test, y_pred_svm)
precision_svm = precision_score(y_test, y_pred_svm)
recall_svm = recall_score(y_test, y_pred_svm)
f1_svm = f1_score(y_test, y_pred_svm)
conf_matrix_svm = confusion_matrix(y_test, y_pred_svm)

print(f"SVM Accuracy: {accuracy_svm:.2f}")
print(f"SVM Precision: {precision_svm:.2f}")
print(f"SVM Recall: {recall_svm:.2f}")
print(f"SVM F1 Score: {f1_svm:.2f}")
print("SVM Confusion Matrix:")
print(conf_matrix_svm)


[I 2023-12-16 01:23:02,763] A new study created in memory with name: no-name-06e65a1f-3914-49b0-86b1-8365a95855b0
